# Session Activity Forecasting with Machine Learning

## Project Overview

This project forecasts future listening session activity using the Last.fm 1K listening history dataset.

The workflow reconstructs listening sessions from raw event data using a 20-minute inactivity threshold, selects the user with the highest number of sessions, and applies supervised machine learning to forecast the next 3 months of session activity.

The project combines PySpark-based preprocessing and feature engineering with a scikit-learn regression pipeline inside a Databricks environment.


## Dataset

Dataset source: [Last.fm 1K Dataset](http://ocelma.net/MusicRecommendationDataset/lastfm-1K.html)

Dataset attribution: This dataset requires referencing the official Last.fm webpage: [Last.fm](https://www.last.fm/)

**The main file used is**: userid-timestamp-artid-artname-traid-traname.tsv

## Methodology
- Reconstruct sessions using a 20-minute inactivity threshold
- Select the user with the highest number of sessions
- Aggregate monthly session activity
- Apply time-series feature engineering
- Train a regression forecasting pipeline
- Predict the next 3 months of session activity


### Import Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

import pandas as pd
import numpy as np
import builtins

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

### Display data

In [0]:
spark = SparkSession.builder.getOrCreate()

input_path = "dbfs:/Volumes/workspace/default/lastfm_split/"

df = (
    spark.read
    .option("sep", "\t")
    .option("header", False)
    .csv(input_path)
)

df = df.toDF(
    "user_id",
    "timestamp",
    "artist_id",
    "artist_name",
    "track_id",
    "track_name"
)

display(df.limit(5))

user_id,timestamp,artist_id,artist_name,track_id,track_name
user_000256,2008-11-27T15:32:58Z,a96ac800-bfcb-412a-8a63-0a98df600700,Modest Mouse,9e71cf4a-4029-4d22-ab37-bba4a6d4485a,Willful Suspension Of Disbelief
user_000256,2008-11-27T13:49:01Z,3648db01-b29d-4ab9-835c-83f6a5068fe4,Godspeed You! Black Emperor,d1ff09ea-2a4b-4fb3-a9fb-66dd43cf32e2,Bbf3
user_000256,2008-11-27T13:38:09Z,3648db01-b29d-4ab9-835c-83f6a5068fe4,Godspeed You! Black Emperor,581592c7-70ff-4502-b149-474acb69f48d,Moya
user_000256,2008-11-27T13:19:11Z,null,Nas To Heaven,null,Antennas To Heaven
user_000256,2008-11-27T12:55:53Z,null,Nas To Heaven,null,Sleep


### Prepare Clean data

In [0]:
df_clean = (
    df
    .filter(col("user_id").isNotNull())
    .filter(col("timestamp").isNotNull())
    .filter(col("track_name").isNotNull())
    .withColumn("event_time", to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .filter(col("event_time").isNotNull())
    .select("user_id", "event_time", "artist_name", "track_name")
)

df_clean.count()

19150867

### Window functions for analytics and forecasting

In [0]:
window_spec = Window.partitionBy("user_id").orderBy("event_time")

df_sessions = (
    df_clean
    .withColumn(
        "prev_time",
        lag("event_time").over(window_spec)
    )
    .withColumn(
        "new_session",
        when(
            col("prev_time").isNull(),
            1
        ).when(
            (
                unix_timestamp(col("event_time")) -
                unix_timestamp(col("prev_time"))
            ) > 20 * 60,
            1
        ).otherwise(0)
    )
    .withColumn(
        "session_num",
        sum("new_session").over(window_spec)
    )
    .withColumn(
        "session_id",
        concat_ws("_", col("user_id"), col("session_num"))
    )
)

display(df_sessions.limit(10))

user_id,event_time,artist_name,track_name,prev_time,new_session,session_num,session_id
user_000008,2008-12-21T02:41:01.000Z,Kanye West,Heartless,null,1,1,user_000008_1
user_000008,2008-12-21T02:41:36.000Z,Kanye West,Amazing (Feat. Young Jeezy),2008-12-21T02:41:01.000Z,0,1,user_000008_1
user_000008,2008-12-21T02:42:09.000Z,Kanye West,Love Lockdown,2008-12-21T02:41:36.000Z,0,1,user_000008_1
user_000008,2008-12-21T02:42:43.000Z,Kanye West,Paranoid (Feat. Mr. Hudson),2008-12-21T02:42:09.000Z,0,1,user_000008_1
user_000008,2008-12-21T02:43:16.000Z,Kanye West,Robocop,2008-12-21T02:42:43.000Z,0,1,user_000008_1
user_000008,2008-12-21T02:43:50.000Z,Kanye West,Street Lights,2008-12-21T02:43:16.000Z,0,1,user_000008_1
user_000008,2008-12-21T02:44:24.000Z,Kanye West,Bad News,2008-12-21T02:43:50.000Z,0,1,user_000008_1
user_000008,2008-12-21T02:44:58.000Z,Kanye West,See You In My Nightmares,2008-12-21T02:44:24.000Z,0,1,user_000008_1
user_000008,2008-12-21T02:45:31.000Z,Kanye West,Coldest Winter,2008-12-21T02:44:58.000Z,0,1,user_000008_1
user_000008,2008-12-21T02:46:05.000Z,Kanye West,Pinocchio Story (Freestyle Live From Singapore),2008-12-21T02:45:31.000Z,0,1,user_000008_1


### Selecting Top User

 User with the highest number of sessions

In [0]:
session_stats = (
    df_sessions
    .groupBy("user_id", "session_id")
    .agg(
        count("*").alias("tracks_in_session"),
        min("event_time").alias("session_start")
    )
)

display(session_stats.limit(10))

top_user_row = (
    session_stats
    .groupBy("user_id")
    .agg(count("*").alias("total_sessions"))
    .orderBy(col("total_sessions").desc())
    .limit(1)
    .collect()[0]
)

top_user_id = top_user_row["user_id"]
top_user_sessions = top_user_row["total_sessions"]

print("Top user:", top_user_id)
print("Total sessions:", top_user_sessions)

user_id,session_id,tracks_in_session,session_start
user_000008,user_000008_1,1382,2008-12-21T02:41:01.000Z
user_000008,user_000008_2,1,2008-12-22T11:44:32.000Z
user_000008,user_000008_3,5,2008-12-22T13:21:23.000Z
user_000008,user_000008_4,95,2008-12-22T15:19:05.000Z
user_000008,user_000008_5,36,2008-12-23T12:36:31.000Z
user_000008,user_000008_6,3,2008-12-23T16:55:41.000Z
user_000008,user_000008_7,4,2008-12-23T17:51:41.000Z
user_000008,user_000008_8,4,2008-12-23T22:51:47.000Z
user_000008,user_000008_9,1,2008-12-24T11:11:34.000Z
user_000008,user_000008_10,80,2008-12-24T14:38:44.000Z


Top user: user_000833
Total sessions: 6897


### Monthly session Stats

In [0]:
monthly_sessions = (
    session_stats
    .filter(col("user_id") == top_user_id)
    .withColumn("month", date_format(col("session_start"), "yyyy-MM"))
    .groupBy("month")
    .agg(count("*").alias("number_of_sessions"))
    .orderBy("month")
)

display(monthly_sessions)

month,number_of_sessions
2005-02,17
2005-03,44
2005-04,63
2005-05,33
2005-06,89
2005-07,51
2005-08,76
2005-09,65
2005-10,116
2005-11,112


### Feature Engineering and ML Pipeline Construction

In [0]:
pdf = monthly_sessions.toPandas()

pdf["month"] = pd.to_datetime(pdf["month"])
pdf = pdf.sort_values("month").reset_index(drop=True)

pdf["month_index"] = range(len(pdf))
pdf["month_of_year"] = pdf["month"].dt.month

pdf["lag_1"] = pdf["number_of_sessions"].shift(1)
pdf["lag_2"] = pdf["number_of_sessions"].shift(2)
pdf["rolling_3"] = pdf["number_of_sessions"].rolling(3).mean()

pdf["target_next_month_sessions"] = pdf["number_of_sessions"].shift(-1)

model_df = pdf.dropna().reset_index(drop=True)

display(model_df)

month,number_of_sessions,month_index,month_of_year,lag_1,lag_2,rolling_3,target_next_month_sessions
2005-04-01T00:00:00.000Z,63,2,4,44.0,17.0,41.333333333333336,33.0
2005-05-01T00:00:00.000Z,33,3,5,63.0,44.0,46.666666666666664,89.0
2005-06-01T00:00:00.000Z,89,4,6,33.0,63.0,61.666666666666664,51.0
2005-07-01T00:00:00.000Z,51,5,7,89.0,33.0,57.666666666666664,76.0
2005-08-01T00:00:00.000Z,76,6,8,51.0,89.0,72.0,65.0
2005-09-01T00:00:00.000Z,65,7,9,76.0,51.0,64.0,116.0
2005-10-01T00:00:00.000Z,116,8,10,65.0,76.0,85.66666666666667,112.0
2005-11-01T00:00:00.000Z,112,9,11,116.0,65.0,97.66666666666667,121.0
2005-12-01T00:00:00.000Z,121,10,12,112.0,116.0,116.33333333333333,123.0
2006-01-01T00:00:00.000Z,123,11,1,121.0,112.0,118.66666666666667,97.0


###  Regression Pipeline Training

In [0]:
feature_cols = [
    "month_index",
    "month_of_year",
    "lag_1",
    "lag_2",
    "rolling_3"
]

X = model_df[feature_cols]
y = model_df["target_next_month_sessions"]

split_index = int(len(model_df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

pipeline.fit

(X_train, y_train)

predictions = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 23.642
RMSE: 33.20828119611131


### Forecast Generation for the Next 3 Months

In [0]:
pipeline.fit(X, y)

future_df = pdf.copy()
forecast_results = []

last_month = future_df["month"].max()

for i in range(1, 4):
    next_month = last_month + pd.DateOffset(months=i)
    next_month_index = future_df["month_index"].max() + 1

    lag_1 = future_df["number_of_sessions"].iloc[-1]
    lag_2 = future_df["number_of_sessions"].iloc[-2]
    rolling_3 = future_df["number_of_sessions"].tail(3).mean()

    next_features = pd.DataFrame([{
        "month_index": next_month_index,
        "month_of_year": next_month.month,
        "lag_1": lag_1,
        "lag_2": lag_2,
        "rolling_3": rolling_3
    }])

    predicted_sessions = pipeline.predict(next_features)[0]
    predicted_sessions = builtins.max(0, int(np.round(predicted_sessions)))

    forecast_results.append({
        "month": next_month.strftime("%Y-%m"),
        "forecast_number_of_sessions": predicted_sessions
    })

    new_row = {
        "month": next_month,
        "number_of_sessions": predicted_sessions,
        "month_index": next_month_index
    }

    future_df = pd.concat([future_df, pd.DataFrame([new_row])], ignore_index=True)

forecast_df = pd.DataFrame(forecast_results)

display(forecast_df)

month,forecast_number_of_sessions
2009-06,203
2009-07,190
2009-08,181


### Save Results

In [0]:
output_path = "dbfs:/Volumes/workspace/default/lastfm/forecast_next_3_months"

forecast_spark = spark.createDataFrame(forecast_df)

(
    forecast_spark
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("sep", "\t")
    .option("header", True)
    .csv(output_path)
)

print(f"Forecast saved successfully to: {output_path}")

Forecast saved successfully to: dbfs:/Volumes/workspace/default/lastfm/forecast_next_3_months
